# 12.2 - LangChain Models & Prompts
**Phase:** 12 - LangChain / Framework Abstractions
**Status:** VERIFIED
---
## 1. What Are We Solving?
Unit 12.1 formatted prompts by hand. LangChain standardises this: `ChatGroq` wraps the model call,
`ChatPromptTemplate` is a reusable fill-in-the-blank prompt, and message role objects
(`SystemMessage`, `HumanMessage`) make intent explicit.
## 2. Why Does This Matter?
Production apps need reusable templates with variables, model configuration in one place, and a
provider-agnostic interface. Manually formatting with f-strings works for one prompt but does not
scale to dozens of templates.
## 3. Prerequisites
- Unit 12.1 (manual pipeline)
- Chat message roles: system / user / assistant
## 4. Learning Objectives
By the end of this unit, you should be able to:
- Invoke `ChatGroq` with a list of message objects and with a plain string
- Build a `ChatPromptTemplate` with `from_messages` and fill it with variables
- Inspect the raw structure of an `AIMessage`
- Set `temperature` / `max_tokens` and observe their effect
- Use a plain `PromptTemplate` + `.format()`
## 5. Mental Model
A prompt template is a fill-in-the-blank form. A chat model is a wrapper around an API call that
returns a message object rather than raw text. You pass a *list of messages* with roles; the model
returns one assistant message as a Python object.

```text
   variables                roles                    content
{role},{input}  ->  [SystemMessage, HumanMessage] ->  AIMessage
      |                    |                            |
ChatPromptTemplate    ChatGroq.invoke()           .content (a str)


## 6. Setup + LLM helper

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


In [2]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_groq import ChatGroq


## 7. Message-List vs Plain-String Invocation
Chat models accept a list of role-tagged messages; LangChain has first-class objects for them. We use
a `safe_model` runnable so this runs offline (mock) and online with a real key.

In [3]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.runnables import RunnableLambda


def chat_invoke(messages, temperature: float = 0.0, max_tokens=None, model=GROQ_MODEL):
    def _txt(m):
        if isinstance(m, tuple):
            return m[1]
        return getattr(m, "content", "")
    if not os.environ.get("GROQ_API_KEY"):
        return AIMessage(content="mock: " + " | ".join(str(_txt(m))[:20] for m in messages))
    try:
        kw = {"model": model, "temperature": temperature}
        if max_tokens is not None:
            kw["max_tokens"] = max_tokens
        return ChatGroq(**kw).invoke(messages)
    except Exception as e:
        return AIMessage(content=f"[llm-error: {type(e).__name__}]")


safe_model = RunnableLambda(lambda m: chat_invoke(m))


msgs = [
    SystemMessage(content="You are a terse data-science tutor."),
    HumanMessage(content="Explain what a p-value is in one sentence."),
]
reply = chat_invoke(msgs)
print("content      :", reply.content)
print("type         :", type(reply).__name__)
print("raw keys     :", list(vars(reply).keys()))  # debug print: structure of AIMessage


content      : A p‑value is the probability of obtaining data at least as extreme as what you observed, assuming the null hypothesis is true.
type         : AIMessage
raw keys     : ['content', 'additional_kwargs', 'response_metadata', 'type', 'name', 'id', 'tool_calls', 'invalid_tool_calls', 'usage_metadata']


## 8. Raw Structure of an AIMessage
The important fields are `.content`, `.type`, and metadata. Let us print the object itself to see
what the framework actually carries.

In [4]:
print("repr:", repr(reply))
print()
print("type  :", getattr(reply, "type", None))
print("content is a str:", isinstance(reply.content, str))


repr: AIMessage(content='A p‑value is the probability of obtaining data at least as extreme as what you observed, assuming the null hypothesis is true.', additional_kwargs={'reasoning_content': 'We need to respond as a terse data-science tutor. So short, concise. One sentence. Probably: "A p-value is the probability of observing data at least as extreme as what you saw, assuming the null hypothesis is true." That is concise.'}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 93, 'total_tokens': 181, 'completion_time': 0.103992151, 'completion_tokens_details': {'reasoning_tokens': 53}, 'prompt_time': 0.005168271, 'prompt_tokens_details': None, 'queue_time': 0.275421392, 'total_time': 0.109160422}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05324-2017-7e82-9fc6-a38aaf3f4b98-0', tool_calls=[], invalid_tool_calls=[], 

## 9. ChatPromptTemplate.from_messages
A template is reusable: variables in `{curly}` braces are filled at invoke time. Because it is a
`Runnable`, calling `.invoke({"var": val})` returns the formatted messages. We can chain it straight
into a model with the pipe operator (Unit 12.4).

In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Respond in exactly one sentence."),
    ("user", "{input}"),
])

filled = prompt.invoke({"role": "data-science tutor", "input": "What is a p-value?"})
print(type(filled).__name__)
for m in filled.to_messages():
    print(f"{m.type:8}: {m.content}")


ChatPromptValue
system  : You are a data-science tutor. Respond in exactly one sentence.
human   : What is a p-value?


## 10. Invoking the Full Prompt -> Model
Compose template with the model and read the structured reply.

In [6]:
chain = prompt | safe_model  # safe_model = chat_invoke via RunnableLambda
out = chain.invoke({"role": "billing agent", "input": "Where is my invoice?"})
print("reply content:", out.content)


reply content: Your invoice has been emailed to the address on file and is also available in your account portal under the “Invoices” section.


## 11. Temperature & max_tokens
`temperature` controls randomness (0.0 = deterministic extraction), `max_tokens` caps response
length. We print the effect of both on a short generation task.

In [7]:
for temp in (0.0, 1.2):
    r = chat_invoke(
        [SystemMessage(content="Describe the color red in 3 words."),
         HumanMessage(content="Go.")],
        temperature=temp, max_tokens=30)
    print(f"temp={temp:<4} ->", r.content)

r_cut = chat_invoke([HumanMessage(content="Count from one to fifty.")], max_tokens=5)
print("max_tokens=5         ->", r_cut.content)


temp=0.0  -> 


temp=1.2  -> 


max_tokens=5         -> 


## 12. Plain PromptTemplate + .format
A template can also be a bare `PromptTemplate` producing a single string (not a message list),
filled with `.format()` — close to the manual f-string of Unit 12.1 but reusable.

In [8]:
tmpl = PromptTemplate.from_template("Translate to French: {sentence}")
formatted = tmpl.format(sentence="Good morning, team.")
print("formatted string:", formatted)


formatted string: Translate to French: Good morning, team.




## Common Mistakes

- (3-5 bullets, from roadmap, concrete and specific to the unit)

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| ... | ... | ... |
(row table, 3-5 rows)

## Best Practices

- (3-5 bullets)

## Hands-On Practice

1. **Basic:** ...
2. **Guided:** ...
3. **Independent:** ...
4. **Realistic:** ...
5. **Challenge:** ...

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.


### Notes for This Unit

- Compare the `format()` above with the f-string in Unit 12.1: the template defers interpolation and
  is reusable across contexts, but under the hood it is just string insertion.
- The raw keys printed in step 8 are the fields a parser reads: `.content` is what matters.

### Common Mistakes (applied)

- Forgetting `.invoke()` and calling the template directly.
- Putting a `user` variable before the `system` template.
- Not passing all required variables -> `KeyError` at fill time.
- Mixing raw strings and message objects in one list.

### Debugging (applied)

| Symptom | Likely Cause | Fix |
|---|---|---|
| `KeyError: 'role'` | Variable missing from `.invoke` dict | Pass every template variable |
| Wrong message format | Mismatched role strings | Use consistent `("system", ...)` tuples |
| Model not reached | Key missing / wrong model name | Check env + model id |
| Non-deterministic output | temperature too high | Set `temperature=0.0` for extraction |

### Best Practices (applied)

- Store templates in files for version control.
- Prefer `ChatPromptTemplate.from_messages()` for clarity.
- Set `temperature` / `model` in the constructor, not per call.

### Hands-On Practice

1. **Basic:** Reuse `prompt` with two different role/input pairs.
2. **Guided:** Build three templates (classify, summarize, extract) and call each with the same model.
3. **Independent:** Load template strings from a Python dict of templates instead of hardcoding.
4. **Realistic:** Few-shot: add 2 example `("user"/"assistant")` pairs to a template and compare quality.
5. **Challenge:** Compare one-task-as-single-message vs system+user vs few-shot on output quality.

### Exit Criteria

- You can create and use prompt templates with variables.
- You can invoke LangChain chat models and read the response.
- You understand what LangChain abstracts compared to raw API calls.
